In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
# Load the Excel file
file_path = 'International_LPI_from_2007_to_2023_Cleaned.xlsx'

In [ ]:
# Load all sheets into a dictionary of DataFrames

sheets_dict = pd.read_excel(file_path, sheet_name=None)

In [ ]:
# Combine all sheets into one DataFrame
df = pd.concat(sheets_dict.values(), ignore_index=True)

In [ ]:
## Individual Single Linear regression on each subscore across all years.

subscores = [
    'Customs Score', 'Infrastructure Score', 'International Shipments Score',
    'Logistics Competence and Quality Score', 'Tracking and Tracing Score', 'Timeliness Score'
]

results = []
for col in subscores:
    X = sm.add_constant(df[col])
    y = df['LPI Score']
    model = sm.OLS(y, X, missing='drop').fit()
    
    results.append({
        'Predictor': col,
        'Coefficient': model.params[col],
        'P-value': model.pvalues[col],
        'R-squared': model.rsquared
    })

results_df = pd.DataFrame(results).sort_values('R-squared', ascending=False)
display(results_df)


In [ ]:
## Individual Single Linear regression on each subscore just for 2023

# Filter dataset for 2023
df_2023 = df[df['Year'] == 2023].copy()

target = 'LPI Score'

# List of sub-score columns
subscores = [
    'Customs Score', 'Infrastructure Score', 'International Shipments Score',
    'Logistics Competence and Quality Score', 'Tracking and Tracing Score', 'Timeliness Score'
]

results = []

for col in subscores:
    X = sm.add_constant(df_2023[col])
    y = df_2023[target]
    model = sm.OLS(y, X, missing='drop').fit()
    
    results.append({
        'Predictor': col,
        'Coefficient': model.params[col],
        'P-value': model.pvalues[col],
        'R-squared': model.rsquared
    })

results_df = pd.DataFrame(results).sort_values('R-squared', ascending=False)
display(results_df)


In [ ]:
## Multilinear regression using all subscores to predict LPI for 2023 across all countries

# Filter for 2023 data
df_2023 = df[df['Year'] == 2023].copy()

# Define dependent and independent variables
target = 'LPI Score'
subscores = [
    'Customs Score', 'Infrastructure Score', 'International Shipments Score',
    'Logistics Competence and Quality Score', 'Tracking and Tracing Score', 'Timeliness Score'
]

# Drop missing data
model_data = df_2023[[target] + subscores].dropna()

# Define X and y
X = model_data[subscores]
y = model_data[target]

# Add intercept
X = sm.add_constant(X)

# Fit model
model = sm.OLS(y, X).fit()

# Show results
print(model.summary())


The model explained 99.7% of the variance in LPI (R² = 0.997, F(6,132) = 8112, p < 0.001). All six sub-scores were statistically significant predictors (p < 0.001), each contributing positively to the overall index. The largest standardized coefficients were found for International Shipments (β = 0.20) and Customs (β = 0.19), suggesting these dimensions play a slightly greater role in differentiating countries’ logistics performance. Model diagnostics indicated no serious violations of regression assumptions.

In [ ]:
## Correlation  Heatmap

# Filter for Mexico
mexico_df = df[df["Country"].str.contains("Mexico", case=False, na=False)]

# ---- Select relevant columns ----
score_cols = [
    "LPI Score",
    "Customs Score",
    "Infrastructure Score",
    "International Shipments Score",
    "Logistics Competence and Quality Score",
    "Timeliness Score",
    "Tracking and Tracing Score"
]

# Compute correlation matrix
corr_matrix = df[score_cols].corr()

# Plot the correlation heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap: LPI and Sub-scores for Mexico", fontsize=14, pad=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
## Corelation Scatter plot Matrix

# Define sub-scores
sub_scores = [
    "Customs Score",
    "Infrastructure Score",
    "International Shipments Score",
    "Logistics Competence and Quality Score",
    "Timeliness Score",
    "Tracking and Tracing Score"
]

# Create scatter plot matrix
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(sub_scores):
    sns.scatterplot(
        data=df,
        x=col,
        y="LPI Score",
        ax=axes[i],
        color="steelblue",
        edgecolor="k",
        s=70
    )
    axes[i].set_title(f"LPI Score vs {col}", fontsize=10)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("LPI Score")

plt.suptitle("Scatter Plot Matrix: LPI Score vs Each Sub-score", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
## Times Series Forecasting - EMS

# Filter for Mexico 
mexico_df = df[df["Country"].str.contains("Mexico", case=False, na=False)]

# Prepare the time series
ts = mexico_df[["Year", "LPI Score"]].dropna().sort_values("Year")
ts = ts.set_index("Year")

# Fit Exponential Smoothing (Holt’s Linear Trend)
model = ExponentialSmoothing(ts, trend="add", seasonal=None)
fit = model.fit(optimized=True)

# Define forecast years
forecast_years = [2025, 2027, 2029, 2031, 2033]
forecast_steps = len(forecast_years)

# Forecast future LPI scores
forecast = fit.forecast(steps=forecast_steps)
forecast.index = forecast_years  # assign forecast years manually

# Plot historical, fitted, and forecast values
plt.figure(figsize=(10, 6))
plt.plot(ts, label="Historical LPI Score", marker='o', color='steelblue')
plt.plot(fit.fittedvalues, label="Fitted Values", linestyle='--', color='orange')
plt.plot(forecast, label="Forecast (2025–2033)", marker='x', linestyle='--', color='red')

plt.title("Mexico LPI Score Forecast (Exponential Smoothing)", fontsize=14)
plt.xlabel("Year")
plt.ylabel("LPI Score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Print forecasted 
print("Forecasted LPI Scores for Future Years:")
print(forecast)

Since LPI data is roughly every 2 years in the data, I assigned the next 5 periods for 2 year intervals. Mexico is projected to continue to follow this downwards trend it has been showing.

In [ ]:
# Filter for Mexico
mexico_df = df[df["Country"].str.contains("Mexico", case=False, na=False)]

# Prepare the time series 
ts = mexico_df[["Year", "LPI Score"]].dropna().sort_values("Year")
ts = ts.set_index("Year")["LPI Score"]

# Fit ARIMA model (using common parameters p=1, d=1, q=1)
model = ARIMA(ts, order=(1, 1, 1))
fit = model.fit()

# Forecast next 5 periods (assign years manually)
forecast_years = [2025, 2027, 2029, 2031, 2033]
forecast_steps = len(forecast_years)

forecast_result = fit.get_forecast(steps=forecast_steps)
forecast_mean = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int()

# Assign forecast index to years
forecast_mean.index = forecast_years
forecast_ci.index = forecast_years

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(ts.index, ts, label="Historical LPI Score", marker='o', color='steelblue')
plt.plot(forecast_mean.index, forecast_mean, label="Forecast (2025–2033)", marker='x', linestyle='--', color='red')
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0],
                 forecast_ci.iloc[:, 1],
                 color='pink', alpha=0.3, label="95% Confidence Interval")

plt.title("Mexico LPI Score Forecast (ARIMA(1,1,1))", fontsize=14)
plt.xlabel("Year")
plt.ylabel("LPI Score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Print forecasted values
print("Forecasted LPI Scores (with 95% confidence intervals):")
print(pd.concat([forecast_mean, forecast_ci], axis=1).rename(
    columns={0: "Lower CI", 1: "Upper CI"}
))

## Mexico follows a downwards projection using the ARIMA method as well, however not as an agressive projection, this makes sense considering Exponentional Smoothing puts higher weights on more recent data points, adn Mexico's most recent points showed large fall off. I would opt to use ARIMA over EMS for forecasting this dataset for a singular country.